In [3]:
import importlib
import pandas as pd
import shared.br_pv_helpers as br_pv
importlib.reload(br_pv)

#build clients and request

from app.weather.open_meteo import OpenMeteoProvider
from app.weather.adapters.default import DefaultWeatherAdapter
from app.weather.open_meteo_mapping import OPEN_METEO_VARIABLES
from app.weather.resolver import WeatherLocationResolver
from app.weather.request import WeatherForecastRequest
from app.weather.result import WeatherForecastResult
from app.forecasting.enums import ForecastMetric
from app.weather.providers.open_meteo.client import Client
from app.weather.location import WeatherLocation

from datetime import timezone, datetime, timedelta

# configuring open meteo client to get historically correct data
api_url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
provider = OpenMeteoProvider(Client(api_url))

resolver = WeatherLocationResolver(
        grid_size=0.1
    )

adapter = DefaultWeatherAdapter(OPEN_METEO_VARIABLES, resolver)

# mimicking WeatherService
def get_forecast(request: WeatherForecastRequest) -> WeatherForecastResult:
    provider_result = (
        provider.get_forecast(
            request
        )
    )
    return adapter.adapt(
        provider_result,
        request
    )

request_variables = [
        ForecastMetric.WIND_SPEED,
        ForecastMetric.CLOUD_COVER,
        ForecastMetric.TEMPERATURE,
        ForecastMetric.GLOBAL_SOLAR_IRRADIANCE,
        ForecastMetric.DIRECT_NORMAL_IRRADIANCE,
        ForecastMetric.DIFFUSE_IRRADIANCE
    ]

In [2]:
# test request
start_date = datetime.now(timezone.utc) - timedelta(seconds=365*24*60*60)

request = WeatherForecastRequest(
        start=start_date,
        end=start_date + timedelta(days=1),
        resolution=timedelta(minutes=15),
        locations=[WeatherLocation(52,10)],
        variables=request_variables
    )
print(get_forecast(request))

WeatherForecastResult(provider='open_meteo', model='best_match', forecasts=[WeatherLocationForecast(location=WeatherLocation(latitude=52.0, longitude=10.0), run=TimeSeriesTimeBase(start=datetime.datetime(2025, 10, 31, 0, 0, tzinfo=datetime.timezone.utc), resolution=datetime.timedelta(seconds=900), slots=97), series=[TimeSeries(metric=<ForecastMetric.WIND_SPEED: 'wind_speed'>, values=[TimeSeriesValue(values=(None, 8.350137710571289, None), value_type='forecast'), TimeSeriesValue(values=(None, 8.70723819732666, None), value_type='forecast'), TimeSeriesValue(values=(None, 9.028754234313965, None), value_type='forecast'), TimeSeriesValue(values=(None, 9.36691951751709, None), value_type='forecast'), TimeSeriesValue(values=(None, 9.726664543151855, None), value_type='forecast'), TimeSeriesValue(values=(None, 9.726664543151855, None), value_type='forecast'), TimeSeriesValue(values=(None, 10.086426734924316, None), value_type='forecast'), TimeSeriesValue(values=(None, 9.726664543151855, None)

In [5]:
# Actual Call to weather forecasts

import time
forecasts = {}
measurements_df = br_pv.get_measurements_df()
metadata_df = br_pv.get_metadata_df()
for ps_id in measurements_df.index.get_level_values("ps_id").unique():
    filtered_measurement = br_pv.filter_for_ps_id(measurements_df, ps_id)
    if len(filtered_measurement) == 0:
        continue
        
    area = br_pv.calc_area_m2(metadata_df, ps_id)
    # get forecast for whole grid in state. used to estimate actual location of asset 
    min_lat, max_lat, min_lon, max_lon = br_pv.get_bounding_rect_for_ps_id(metadata_df, ps_id)
    
    # create 4x4 grid
    grid_count = 4
    locations = []
    for i_lat in range(0, grid_count):
        lat = min_lat + i_lat*(max_lat - min_lat)/(grid_count - 1)
        for i_lon in range(0, grid_count):
            lon = min_lon + i_lon*(max_lon - min_lon)/(grid_count - 1)
            locations.append(WeatherLocation(lat,lon))

    # request for all grid locations and for all dates in measurement data
    start_date = filtered_measurement.index[0][0]
    end_date = filtered_measurement.index[len(filtered_measurement)-1][0]
    request = WeatherForecastRequest(
        start=start_date,
        end=end_date,
        resolution=timedelta(minutes=15),
        locations=locations,
        variables=request_variables
    )
    # save for later use 
    forecasts[ps_id] = get_forecast(request)
    # simple protection against api request limit
    print("Successfully requested weather for ps_id ", ps_id, (min_lat, max_lat, min_lon, max_lon))
    time.sleep(15)

Successfully requested weather for ps_id  PS_001 (-25.31, -19.78, -53.11, -44.16)


In [6]:
# check for plausible result data
print(forecasts['PS_001'].forecasts[0].location)
print(forecasts['PS_001'].forecasts[0].run)
print(forecasts['PS_001'].forecasts[0].series[0].values[0:20])
print(forecasts['PS_001'].forecasts[0].series[1].metric)
print(forecasts['PS_001'].forecasts[4].location)
print(forecasts['PS_001'].forecasts[5].location)
print(forecasts.keys())

WeatherLocation(latitude=-25.3, longitude=-53.1)
TimeSeriesTimeBase(start=datetime.datetime(2024, 3, 26, 0, 0, tzinfo=datetime.timezone.utc), resolution=datetime.timedelta(seconds=900), slots=42249)
[TimeSeriesValue(values=(None, 5.116561412811279, None), value_type='forecast'), TimeSeriesValue(values=(None, 5.9834771156311035, None), value_type='forecast'), TimeSeriesValue(values=(None, 7.309418678283691, None), value_type='forecast'), TimeSeriesValue(values=(None, 8.686748504638672, None), value_type='forecast'), TimeSeriesValue(values=(None, 9.387651443481445, None), value_type='forecast'), TimeSeriesValue(values=(None, 9.387651443481445, None), value_type='forecast'), TimeSeriesValue(values=(None, 8.686748504638672, None), value_type='forecast'), TimeSeriesValue(values=(None, 7.956732749938965, None), value_type='forecast'), TimeSeriesValue(values=(None, 7.145795822143555, None), value_type='forecast'), TimeSeriesValue(values=(None, 6.49248743057251, None), value_type='forecast'), 

In [7]:
# build dataframe from forecasts

df_rows = {}
locations = {}
locations_df_dict = {}
for ps_id, forecast_result in forecasts.items():
    locations[ps_id] = {}
    for lid, forecast in enumerate(forecast_result.forecasts):
        start = forecast.run.start
        resolution = forecast.run.resolution
        # save actual locations of forecast result
        locations[ps_id][lid] = forecast.location

        lat_col = 'lat_' + str(lid)
        lon_col = 'lon_' + str(lid)
        if lat_col not in locations_df_dict.keys():
            locations_df_dict[lat_col] = {}
        if lon_col not in locations_df_dict.keys():
            locations_df_dict[lon_col] = {}
        locations_df_dict[lat_col][ps_id] = forecast.location.latitude
        locations_df_dict[lon_col][ps_id] = forecast.location.longitude

        # save all values and series in pre-dataframe dict
        for sid, series in enumerate(forecast.series):
            column_name = series.metric + '_' + str(lid)
            time = start
            if column_name not in df_rows.keys():
                    df_rows[column_name] = {}
                
            for value in series.values:
                index = (time, ps_id)
                df_rows[column_name][index] = value.values[1] # hackish access to p50 value should be series.quantile(value,0.5)
                time += resolution
                
                



In [8]:
# Save dataframes
df = pd.DataFrame(df_rows)
# important for easy join
df.index = df.index.set_names(["datetime", "ps_id"])
print(df['cloud_cover_0'])

#df.to_parquet(path + "weather_forecasts_multi_location.parquet")

# br_pv.save_forecast_locations(locations)
locations

datetime                   ps_id 
2024-03-26 00:00:00+00:00  PS_001     1.00
2024-03-26 00:15:00+00:00  PS_001     0.75
2024-03-26 00:30:00+00:00  PS_001     0.50
2024-03-26 00:45:00+00:00  PS_001     0.25
2024-03-26 01:00:00+00:00  PS_001     0.00
                                     ...  
2025-06-09 22:45:00+00:00  PS_001    11.50
2025-06-09 23:00:00+00:00  PS_001    15.00
2025-06-09 23:15:00+00:00  PS_001    15.00
2025-06-09 23:30:00+00:00  PS_001    15.00
2025-06-09 23:45:00+00:00  PS_001    15.00
Name: cloud_cover_0, Length: 42336, dtype: float64


{'PS_001': {0: WeatherLocation(latitude=-25.3, longitude=-53.1),
  1: WeatherLocation(latitude=-25.3, longitude=-50.1),
  2: WeatherLocation(latitude=-25.3, longitude=-47.1),
  3: WeatherLocation(latitude=-25.3, longitude=-44.2),
  4: WeatherLocation(latitude=-23.400000000000002, longitude=-53.1),
  5: WeatherLocation(latitude=-23.400000000000002, longitude=-50.2),
  6: WeatherLocation(latitude=-23.400000000000002, longitude=-47.1),
  7: WeatherLocation(latitude=-23.400000000000002, longitude=-44.2),
  8: WeatherLocation(latitude=-21.6, longitude=-53.2),
  9: WeatherLocation(latitude=-21.6, longitude=-50.1),
  10: WeatherLocation(latitude=-21.6, longitude=-47.2),
  11: WeatherLocation(latitude=-21.6, longitude=-44.1),
  12: WeatherLocation(latitude=-19.8, longitude=-53.1),
  13: WeatherLocation(latitude=-19.8, longitude=-50.2),
  14: WeatherLocation(latitude=-19.8, longitude=-47.1),
  15: WeatherLocation(latitude=-19.8, longitude=-44.1)}}